# FasiAI — Home Price Model (Gradient Boosting)

Gradient-boosted regression on the King County housing data with feature engineering, evaluation, and plots.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

## Load

In [ ]:
try:
    df = pd.read_csv("data/source/usa_housing.csv")
except FileNotFoundError:
    df = pd.read_csv("/Users/demetre/Downloads/USA Housing Dataset.csv")

df = df[(df["price"] > 0) & (df["sqft_living"] > 0)].reset_index(drop=True)
df.shape

## Preprocessing & feature engineering

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["sale_year"] = df["date"].dt.year
df["sale_month"] = df["date"].dt.month
df["house_age"] = (df["sale_year"] - df["yr_built"]).clip(lower=0)
df["renovated"] = (df["yr_renovated"] > 0).astype(int)
df["years_since_update"] = (df["sale_year"] - df[["yr_built", "yr_renovated"]].max(axis=1)).clip(lower=0)
df["has_basement"] = (df["sqft_basement"] > 0).astype(int)
df["bath_per_bed"] = df["bathrooms"] / df["bedrooms"].replace(0, 1)
df["living_lot_ratio"] = df["sqft_living"] / df["sqft_lot"].clip(lower=1)

In [ ]:
base = [
    "bedrooms", "bathrooms", "sqft_living", "sqft_lot", "floors",
    "waterfront", "view", "condition", "sqft_above", "sqft_basement",
    "house_age", "renovated", "years_since_update", "has_basement",
    "bath_per_bed", "living_lot_ratio", "sale_year", "sale_month",
]
city = pd.get_dummies(df["city"], prefix="city").astype(int)
X = pd.concat([df[base], city], axis=1)
y = np.log1p(df["price"])

## Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Model

In [ ]:
try:
    from xgboost import XGBRegressor
    model = XGBRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
    )
except Exception:
    from sklearn.ensemble import GradientBoostingRegressor
    model = GradientBoostingRegressor(
        n_estimators=900,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        random_state=42,
    )

model.fit(X_train, y_train)

## Evaluation

In [ ]:
pred = np.expm1(model.predict(X_test))
actual = np.expm1(y_test.values)

mse = mean_squared_error(actual, pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(actual, pred)
r2 = r2_score(actual, pred)
median_ape = np.median(np.abs(pred - actual) / actual) * 100

print("MSE:  {:,.0f}".format(mse))
print("RMSE: {:,.0f}".format(rmse))
print("MAE:  {:,.0f}".format(mae))
print("R2:   {:.3f}".format(r2))
print("Median APE %: {:.1f}".format(median_ape))

## Plots

In [ ]:
resid = pred - actual
gold = "#C9A227"
fig, ax = plt.subplots(2, 2, figsize=(13, 10))

ax[0, 0].scatter(actual, pred, s=10, alpha=0.4, color=gold)
lims = [actual.min(), actual.max()]
ax[0, 0].plot(lims, lims, color="#444", linestyle="--")
ax[0, 0].set_xlabel("Actual price")
ax[0, 0].set_ylabel("Predicted price")
ax[0, 0].set_title("Predicted vs actual")

ax[0, 1].hist(resid, bins=60, color=gold)
ax[0, 1].set_xlabel("Residual (pred - actual)")
ax[0, 1].set_title("Residual distribution")

ax[1, 0].scatter(pred, resid, s=10, alpha=0.4, color=gold)
ax[1, 0].axhline(0, color="#444", linestyle="--")
ax[1, 0].set_xlabel("Predicted price")
ax[1, 0].set_ylabel("Residual")
ax[1, 0].set_title("Residuals vs predicted")

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False).head(15)
ax[1, 1].barh(importances.index[::-1], importances.values[::-1], color=gold)
ax[1, 1].set_title("Top 15 feature importances")

plt.tight_layout()
plt.show()